## **Feature Engineering**

This section describes the feature engineering pipeline used to transform raw transaction data into structured inputs for machine learning models. Unlike traditional preprocessing, this pipeline is designed to:

- Capture **user behavioral patterns**
- Detect **anomalies and outliers**
- Incorporate **domain-specific risk signals**
- Ensure **consistency between training and production environments**

The resulting feature space represents a combination of:
- Statistical transformations
- Behavioral indicators
- Context-aware risk metrics

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


## **Data Preparation & Train/Test split**

Raw data is cleaned and standardized before feature engineering:

- Removed irrelevant or high-cardinality columns: `fraud_pattern`, `transaction_id`, `account_id`
- Converted `timestamp` to datetime format
- Dropped invalid timestamps and sorted data chronologically

This ensures data consistency and prepares the dataset for time-dependent processing.


Data is split chronologically instead of randomly to avoid data leakage and reflects real-world deployment, where models are trained on historical data and used to predict future behavior.

In [8]:
target = "is_fraud"

cate_cols = [
    "merchant_category",
    "merchant_country",
    "device_type",
    "mcc_code",
    "hour_of_day",
    "day_of_week",
]

In [17]:
#Load data
def basic_prepare(df):
    df = df.copy()

    df = df.drop(columns=["fraud_pattern", "transaction_id", "account_id"], errors="ignore")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    return df

train = pd.read_csv("../data/csv/train.csv")
test  = pd.read_csv("../data/csv/test.csv")

train = basic_prepare(train)
test  = basic_prepare(test)

In [18]:
#Time-based split
def time_split(df, valid_size=0.2):
    n = len(df)
    split = int(n * (1 - valid_size))
    return df.iloc[:split].copy(), df.iloc[split:].copy()

train_df, valid_df = time_split(train)

## **Safe Division and Safe Logarithm functions**
**Safe Division** performs element-wise division while handling edge cases:

- Replaces zero values in the denominator with a small constant (1e-10) to avoid division by zero
- Converts infinite results to a predefined fallback value
- Fills missing values to maintain numerical consistency

**Safe Logarithm** applies a stable logarithmic transformation:

Uses log1p(x) to handle small values and reduce skewness
Clips negative inputs to zero to ensure valid logarithm computation
Replaces invalid outputs (infinite or missing values) with a fallback value

**Purpose:** These utilities prevent numerical instability during preprocessing, ensuring that:

- Feature transformations do not introduce invalid values (NaN, ±∞)
- Downstream models receive clean and consistent inputs
- The pipeline remains stable in both training and production environments

In [11]:
#Safe Division and Safe Logarithm functions
def safe_divide(num, denom, fill=0):
    return (num / (denom.replace(0, 1e-10))).replace([np.inf,-np.inf], fill).fillna(fill)

def safe_log(s, fill=0):
    return pd.Series(np.log1p(np.maximum(s, 0)), index=s.index).replace([np.inf,-np.inf], fill).fillna(fill)

## **Quantile-based Thresholds**

Thresholds are derived from training data:

- amount_q95 → high-value transactions
- ip_q80 → high-risk IP activity
- ratio_q90, velocity_q90 → abnormal behavior levels

Fraud often occurs in the **extreme ends (tails)** of distributions so using quantiles allows the model to adapt to data scale and detect statistically rare events

In [12]:
# Fit quantile-based parameters for risk amount features
def fit_params(df):
    params = {}

    # Amount
    params['amount_q95'] = df['amount'].quantile(0.95)

    # Risk
    if "ip_risk_score" in df.columns:
        params['ip_q80'] = df['ip_risk_score'].quantile(0.80)

    if "amount_vs_avg_ratio" in df.columns:
        params['ratio_q90'] = df['amount_vs_avg_ratio'].quantile(0.90)

    if "velocity_1h" in df.columns:
        params['velocity_q90'] = df['velocity_1h'].quantile(0.90)

    if "credit_limit" in df.columns:
        util = df['amount'] / df['credit_limit'].replace(0, np.nan)
        params['util_q90'] = util.quantile(0.90)

    print("✅ Quantile params fitted")
    return params

## **Creating Features**

**1. Amount Features**

- log_amount: reduces skewness
- is_round_amount: captures structured values
- high_amount_flag: detects extreme transactions

These features combine normalization and anomaly detection.

**2. Time Features**

- is_night (0–5h)
- is_business_hours (9–17h)

Time-based features provide context, as fraud often occurs at unusual hours.

**3. Risk-based Features**

- ip_risk_score → network risk
- velocity_1h → transaction frequency
- amount_vs_avg_ratio → behavioral deviation

Each feature is log-transformed and thresholded to capture abnormal patterns.

- utilization = amount / credit_limit

This feature measures spending relative to capacity, enabling context-aware risk detection.

In [13]:
#Add features
def add_features(df, params):
    df = df.copy()
    new_var = []
    
    # 1. AMOUNT FEATURES
    if 'amount' in df.columns:
        df['log_amount'] = safe_log(df['amount'])
        df['is_round_amount'] = ((df['amount'] % 10 == 0) & (df['amount'] > 0)).astype(int)
        df['high_amount_flag'] = (df['amount'] > params['amount_q95']).astype(int)

        new_var += ['log_amount', 'is_round_amount', 'high_amount_flag']

    # 2. TIME FEATURES
    if 'hour_of_day' in df.columns:
        
        df['is_night'] = df['hour_of_day'].between(0, 5).astype(int)
        df['is_business_hours'] = df['hour_of_day'].between(9, 17).astype(int)

        new_var += ['is_night', 'is_business_hours']

    # 3. RISK FEATURES
    # 1) IP risk score features
    if "ip_risk_score" in df.columns and 'ip_q80' in params:
        df["log_ip_risk_score"] = safe_log(df["ip_risk_score"])
        df["high_ip_risk_flag"] = (df["ip_risk_score"] > params['ip_q80']).astype(int)
        new_var += ["log_ip_risk_score", "high_ip_risk_flag"]
        df.drop(columns=["ip_risk_score"], inplace=True)

    if "amount_vs_avg_ratio" in df.columns and 'ratio_q90' in params:
        df["log_amount_vs_avg_ratio"] = safe_log(df["amount_vs_avg_ratio"])
        df["high_amount_vs_avg_flag"] = (df["amount_vs_avg_ratio"] > params['ratio_q90']).astype(int)
        new_var += ["log_amount_vs_avg_ratio", "high_amount_vs_avg_flag"]
        df.drop(columns=["amount_vs_avg_ratio"], inplace=True)

    if "velocity_1h" in df.columns and 'velocity_q90' in params:
        df["log_velocity_1h"] = safe_log(df["velocity_1h"])
        df["high_velocity_1h_flag"] = (df["velocity_1h"] > params['velocity_q90']).astype(int)
        new_var += ["log_velocity_1h", "high_velocity_1h_flag"]
        df.drop(columns=["velocity_1h"], inplace=True)

    if "credit_limit" in df.columns and 'util_q90' in params:
        df["utilization"] = safe_divide(df["amount"], df["credit_limit"])
        df["high_utilization_flag"] = (df["utilization"] > params['util_q90']).astype(int)
        new_var += ["utilization", "high_utilization_flag"]
        df.drop(columns=['amount'], inplace=True)
        df.drop(columns=["credit_limit"], inplace=True)

    print(f"✅ Features added: {len(new_var)}")

    return df, new_var

In [19]:
# fit
params = fit_params(train_df)

train_df, _ = add_features(train_df, params)
valid_df, _ = add_features(valid_df, params)
test, _     = add_features(test, params)

✅ Quantile params fitted
✅ Features added: 13
✅ Features added: 13
✅ Features added: 13


In [20]:
def remove_non_model_cols(df):
    df = df.copy()
    df = df.drop(columns=["timestamp"], errors="ignore")
    return df

train_df = remove_non_model_cols(train_df)
valid_df = remove_non_model_cols(valid_df)
test     = remove_non_model_cols(test)

## **Logistic Regression Encode**

This stage prepares the dataset for linear models such as Logistic Regression, which require fully numerical and properly scaled inputs.

Categorical features are transformed using One-Hot Encoding, converting each category into independent binary variables. This prevents the model from assuming any ordinal relationship between categories and allows it to learn unbiased weights for each value.

After encoding, all features are standardized using feature scaling (StandardScaler). This ensures that variables operate on comparable scales, improving convergence during optimization and preventing features with large magnitudes from dominating the model.

In [33]:
num_cols = ['account_age_days', 'time_since_last_s', 'log_amount', 'is_round_amount', 'high_amount_flag', 'log_ip_risk_score', 'high_ip_risk_flag', 'log_amount_vs_avg_ratio', 'high_amount_vs_avg_flag', 'log_velocity_1h', 'high_velocity_1h_flag', 'utilization', 'high_utilization_flag']
cate_cols = ['merchant_category', 'merchant_country', 'device_type', 'mcc_code', 'hour_of_day', 'day_of_week']
binary_cols = ['is_fraud', 'is_weekend', 'card_present', 'device_known', 'is_foreign_txn', 'has_2fa']

**One-Hot Encoding for categorical variables**

Categorical variables are transformed using One-Hot Encoding, converting each category into a binary vector. This avoids imposing artificial ordinal relationships between categories, which is critical for models like Logistic Regression.

The encoder is fitted only on the training set to prevent data leakage.
handle_unknown='ignore' ensures robustness when unseen categories appear in validation/test data.
After encoding:
Original categorical columns are removed
New binary columns are appended

Validation and test sets are processed using the same fitted encoder to maintain feature consistency.

In [ ]:
# One-hot Encode for categorical variables
def fit_onehot(df, cols):
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    df[cols] = df[cols].astype(str)
    encoded = enc.fit_transform(df[cols])

    new_cols = enc.get_feature_names_out(cols)
    df_encoded = pd.DataFrame(encoded, columns=new_cols, index=df.index)

    df = df.drop(columns=cols)
    df = pd.concat([df, df_encoded], axis=1)

    return df, enc, cols

def transform_onehot(df, cols, enc):
    df[cols] = df[cols].astype(str)

    encoded = enc.transform(df[cols])
    new_cols = enc.get_feature_names_out(cols)

    df_encoded = pd.DataFrame(encoded, columns=new_cols, index=df.index)

    df = df.drop(columns=cols)
    df = pd.concat([df, df_encoded], axis=1)

    return df

## **Scale numeric variables**

Numerical features are standardized using StandardScaler:

- Mean = 0
- Standard deviation = 1

This step is essential for linear models because:

- It stabilizes optimization during training
- Prevents features with large magnitudes from dominating the model

Important constraints:

- The scaler is fitted on the training set only
- The same transformation is applied to validation and test sets

In [23]:
#Scale numeric variables
cat_cols = [c for c in cate_cols if c in train_df.columns]

train_log, enc, used_cat = fit_onehot(train_df, cat_cols)

num_cols = [c for c in train_log.columns if c != target]

scaler = StandardScaler()
train_log[num_cols] = scaler.fit_transform(train_log[num_cols])

train_cols = train_log.columns

In [24]:
valid_log = transform_onehot(valid_df, used_cat, enc)
valid_log[num_cols] = scaler.transform(valid_log[num_cols])
valid_log = valid_log.reindex(columns=train_cols, fill_value=0)

test_log = transform_onehot(test, used_cat, enc)
test_log[num_cols] = scaler.transform(test_log[num_cols])
test_log = test_log.reindex(columns=train_cols, fill_value=0)

## **Tree Models Encode**

This stage prepares the dataset for tree-based models (e.g., Random Forest, Gradient Boosting), which can naturally handle categorical features encoded as integers without requiring one-hot expansion.

Categorical variables are transformed using Label Encoding, where each category is mapped to a unique integer. Unlike linear models, tree models do not assume ordinal relationships between these values, so this encoding is both efficient and appropriate.

## Label Encoding

This function converts categorical features into integer representations for use in tree-based models.

- During training, a `LabelEncoder` is fitted for each categorical column and stored.
- During inference, the same encoders are reused to ensure consistency.
- Unseen categories are mapped to a default value (`"UNKNOWN"`) to prevent transformation errors.

### Purpose

- Maintain consistent feature representation across datasets  
- Handle unseen categories robustly  
- Provide an efficient encoding scheme for tree-based models

In [27]:
def label_encode(df, encoders=None, target='is_fraud'):
    df = df.copy()

    if encoders is None:
        encoders = {}
        
        for col in cate_cols:
            
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
            print(f"✅ Fit & encoded '{col}' ({df[col].nunique()} cats)")
    
    else:
        for col in cate_cols:
            if col in encoders:
                le = encoders[col]

                # handle unseen labels
                df[col] = df[col].astype(str)
                df[col] = df[col].map(lambda x: x if x in le.classes_ else 'UNKNOWN')

                # add UNKNOWN to encoder if needed
                if 'UNKNOWN' not in le.classes_:
                    le.classes_ = np.append(le.classes_, 'UNKNOWN')

                df[col] = le.transform(df[col])
            else:
                df[col] = 0  # fallback

    return df, encoders

In [28]:
train_tree, le_dict = label_encode(train_df)
valid_tree, _ = label_encode(valid_df, le_dict)
test_tree, _  = label_encode(test, le_dict)

train_cols = train_tree.columns

valid_tree = valid_tree.reindex(columns=train_cols, fill_value=0)
test_tree  = test_tree.reindex(columns=train_cols, fill_value=0)

✅ Fit & encoded 'merchant_category' (14 cats)
✅ Fit & encoded 'merchant_country' (11 cats)
✅ Fit & encoded 'device_type' (5 cats)
✅ Fit & encoded 'mcc_code' (14 cats)
✅ Fit & encoded 'hour_of_day' (24 cats)
✅ Fit & encoded 'day_of_week' (7 cats)


In [32]:
joblib.dump(params, "fe_params.pkl")

train_log.to_parquet("train_log.parquet", index=False)
valid_log.to_parquet("valid_log.parquet", index=False)
test_log.to_parquet("test_log.parquet", index=False)

train_tree.to_parquet("train_tree.parquet", index=False)
valid_tree.to_parquet("valid_tree.parquet", index=False)
test_tree.to_parquet("test_tree.parquet", index=False)